# 授業用 人狼エージェント — Colab クイックスタート

1. 下のセルでリポジトリを取得します（`REPO` を授業のリポジトリに書き換えてください）。
2. `API` に先生から配られた Apps Script の URL を入れます。
3. `my_agent.py` を編集して対戦します。

In [ ]:
REPO = "https://github.com/<ユーザー名>/<リポジトリ名>.git"  # ← 書き換える
!git clone -q $REPO jinro && echo OK
%cd jinro/agents
!pip -q install requests

In [ ]:
API = "https://script.google.com/macros/s/XXXX/exec"  # ← 先生から配られた URL

## まずはサンプル同士で 3 戦
ルールベース 3 体 + 自作 1 体 + CPU 1 体で対戦し、勝率を表示します。

In [ ]:
!python run_agents.py --api "$API" --create --agent rule:3 --agent my:1 --fill-cpu --games 3 --talk-sec 20 --vote-sec 15 --night-sec 15 --quiet

## 自分のエージェントを編集する
左のファイル一覧から `jinro/agents/my_agent.py` を開いて編集するか、下のセルで上書きします。

In [ ]:
%%writefile my_agent.py
from rule_agent import RuleAgent
from wolfclient import Game, VOTE, ESTIMATE


class MyAgent(RuleAgent):
    def talk(self, g: Game):
        if g.round == 3 and g.my_role != "WEREWOLF":
            t = self.vote(g)
            return f"{g.name(t)}が人狼だと思います。", [ESTIMATE(t, "WEREWOLF")]
        return super().talk(g)


## 人間が待っている村に参加させる
画面の待機室に表示される村の ID を入れてください。

In [ ]:
ROOM = "abc123"  # ← 村の ID
!python run_agents.py --api "$API" --room $ROOM --agent my:1

## LLM エージェント
API キーは左の🔑（シークレット）に `LLM_API_KEY` という名前で登録してください。

In [ ]:
import os
from google.colab import userdata
os.environ["LLM_API_KEY"] = userdata.get("LLM_API_KEY")
os.environ["LLM_BASE_URL"] = "https://api.openai.com/v1"  # Gemini / Claude / Ollama なども可（README 参照）
os.environ["LLM_MODEL"] = ""  # ← 使うモデル名
!python run_agents.py --api "$API" --create --agent llm:1 --agent rule:2 --fill-cpu --games 2 --talk-sec 60